# Perplexity metric

### Prediction:
Knowing nothing about any of this yet, does a model with a perplexity of 239 mean that at each token it hesitates between the 239 most likely ones? I think I'm missing a formula, but I don't know. 

> **Re-run on 7 Nov 2026 with the final adapter** (`run1_5ep` at step 2,500, chosen by early stopping, decision 15), loaded from `adapters/run1_5ep_best`. The predictions are kept as I wrote them, for `run1` (1,000 steps, 0.74 epochs). The results below are the final adapter's, with `run1`'s numbers kept next to them for reference.

After doing the exercises, I understand perplexity much better. Given that the base case is a perplexity of 239, which is a lot but you can tell it has some idea... The normal thing is for it to drop fast in the first iterations. For it to understand, for example, that responses are usually short. With `--mask-prompt`, so that it doesn't take the context into account in the perplexity or loss calculation (which isn't useful to us in this particular case).

After a first test iteration, perplexity dropped to 24, and it hadn't even seen all the examples. The problem is it's still far enough off that we don't know whether what it learned is how I talk or how people talk on WhatsApp in Spanish with my contacts. It's a distinction we have to keep in mind.

I expect a final perplexity of around 15, in a good case, with a couple of passes over all the data. But it'll depend on whether the model manages to understand how I write. Probably yes, I don't think my style is that hard to understand. Though maybe 15 is optimistic. My idea is that hesitating between 15 possible tokens seems reasonable to me on average to consider that it has understood the style. Though the biggest uncertainty would be in the first token of each response, I think.

For run1, I expect a perplexity of around 22. It's a run on the test set, so it should be relatively similar to the validation set, since the sampling is random. Although now that I think about it, these are chats it has never seen. Because we did split whole chats for test. So let's bump the number up a bit. 30.

In [1]:
!cd .. && /usr/bin/time -l uv run mlx_lm.lora --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --adapter-path adapters/run1_5ep_best --data data/dataset/session --mask-prompt --test --test-batches -1

Loading pretrained model
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 3722.02it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
Download complete:                                         |  0.00B            
Loading datasets
Testing
Calculating loss...: 100%|███████████████████| 283/283 [01:20<00:00,  3.51it/s]
Test loss 3.205, Test ppl 24.664.
       84.95 real         7.92 user        17.73 sys
           963919872  maximum resident set size
                   0  average shared memory size
                   0  average unshared data size
                   0  average unshared stack size
              107863  page reclaims
                  10  page faults
                   0  swaps
                   0  block input operations
                   0  block output operatio

### Result:
- Perplexity of 24.7 with mlx's measure (run1 gave 25.5). Very similar to the validation set. It's still me talking even if they're different chats. The problem with this is that we don't know whether the model has learned to talk like me or just to talk in general in Spanish on WhatsApp. We need more tests to disambiguate. I'm also leaving here the memory and time metrics in human format.

### Metrics (final adapter):
| Metric | Value |
|---|---|
| Test loss | 3.205 |
| Test ppl | 24.66 |
| Examples | 1,133 (283 batches) |
| Time | 1 min 25 s (1 min 20 s measuring) |
| Peak memory (RSS) | 964 MB |


## Test baseline
Clearly it should be the same as on test in the first iteration. It's the same kind of messages and with no adaptation at all, it should give a very similar result, close to the original perplexity of 239.

In [2]:
!cd .. && /usr/bin/time -l uv run mlx_lm.lora --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --adapter-path '' --data data/dataset/session --mask-prompt --test --test-batches -1

Loading pretrained model
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 2027.00it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
Download complete:                                         |  0.00B            
Loading datasets
Testing
Calculating loss...: 100%|███████████████████| 283/283 [01:09<00:00,  4.08it/s]
Test loss 5.344, Test ppl 209.248.
       73.87 real         6.08 user        12.28 sys
           941719552  maximum resident set size
                   0  average shared memory size
                   0  average unshared data size
                   0  average unshared stack size
              105654  page reclaims
                  56  page faults
                   0  swaps
                   0  block input operations
                   0  block output operati

### Result:
- Perplexity of 209. Very close to the prediction (small changes at such high levels change the value a lot), so there's hardly any difference. That was to be expected.

| Model | Test loss | Test ppl | Time | Peak memory (RSS) |
|---|---|---|---|---|
| Base (no adapter) | 5.344 | 209.2 | 1 min 14 s | 942 MB |
| Final adapter (LoRA, step 2,500) | 3.205 | 24.7 | 1 min 25 s | 964 MB |
| Difference | -2.139 nats (-40%) | 8.5 times lower | | |

(run1: 3.237, ppl 25.5. Memory now is higher than in the first run, 577 and 614 MB, probably because other jobs were sharing the machine; it is not comparable.)

Training has reduced the surprise (loss) by 40%. It's much closer to communicating the right way. Perplexity went down 8.5 times. So it already has an idea of what kind of responses it should give.


## Friends expectation (control group)
The idea with this test is to check how much the model has really learned from me (to be able to compare) against what it has learned about how people talk on WhatsApp in general with my friends. Here the model has to predict my contacts' responses. Not mine.

If training went well, the perplexity should be higher than mine (25), so I expect it to be around 40, ideally more. If it came out similar, it would mean the model hasn't learned from me, but has learned what gets said in general. Which is also likely, since my way of communicating might be very homogeneous with my friends' and not stand out. Let's check

In [3]:
!cd .. && uv run mlx_lm.lora --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --adapter-path '' --data data/dataset/friends --mask-prompt --test --test-batches -1

Loading pretrained model
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 3397.42it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
Download complete:                                         |  0.00B            
Loading datasets
Testing
Calculating loss...: 100%|███████████████████| 277/277 [01:09<00:00,  4.00it/s]
Test loss 5.298, Test ppl 199.838.


In [4]:
!cd .. && uv run mlx_lm.lora --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --adapter-path adapters/run1_5ep_best --data data/dataset/friends --mask-prompt --test --test-batches -1

Loading pretrained model
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 4587.28it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
Download complete:                                         |  0.00B            
Loading datasets
Testing
Calculating loss...: 100%|███████████████████| 277/277 [01:19<00:00,  3.47it/s]
Test loss 3.683, Test ppl 39.756.


### Result
The base perplexity on my friends is 199.8, with a loss of 5.3. Somewhat better than what we got with my data, suggesting my friends may talk in a way that's more aligned with the base model.

Using the final adapter, the loss and perplexity dropped to 3.68 and 39.8 (run1: 3.665 and 39.1). Right on the prediction. So yes, the model has learned something from me. Relative drops:
Loss drop: 30.5%
PPL drop: 80.1%


## Comparison: does it talk like me or like WhatsApp? (mlx's measure)

| Responses from | Base loss | Final loss | Loss drop | Base PPL | Final PPL | PPL drop |
|---|---|---|---|---|---|---|
| Me (test) | 5.344 | 3.205 | 40.0% | 209.2 | 24.7 | 88.2% |
| My friends (test_friends) | 5.298 | 3.683 | 30.5% | 199.8 | 39.8 | 80.1% |
| **Signal (me - friends)** | | | | | | **8.1 points** (run1: 7.4) |

Same test (77 sessions no model saw), `--mask-prompt`, LoRA rank 8. These numbers use mlx-lm's loss, which has the padding bug explained below.


### Bootstrap
To really compare whether it learns about me or about WhatsApp, we're going to run a bootstrap of 1000 iterations. We'll compare both the loss and the perplexity to have more solid data, even though one depends on the other. For that, the most sensible thing is to compare sessions, since single messages aren't independent of each other, and the bootstrap assumes they are. The only independent format of the two is the session

In [5]:
import json, math, random
import mlx.core as mx
from mlx_lm import load

MODELO = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
model, tokenizer = load(MODELO, adapter_path="../adapters/run1_5ep_best")  # final adapter: run1_5ep at step 2500
base, tok_b = load(MODELO)  # base, no adapter

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [6]:
def nll_ejemplo(model, tokenizer, messages):
    full = tokenizer.apply_chat_template(messages)
    base_prompt = tokenizer.apply_chat_template(
        messages[:-1], add_generation_prompt=True)
    n_tokens = len(full) - len(base_prompt)
    logits = model(mx.array(full)[None])
    logits = logits[0, len(base_prompt)-1:len(full)-1]
    logits = logits.astype(mx.float32)
    logprobs = logits - mx.logsumexp(logits, axis=-1, keepdims=True)
    nll = 0
    for i in range(n_tokens):
        nll -= logprobs[i, full[len(base_prompt) + i]].item()
    mx.clear_cache()
    return (nll, n_tokens)

In [7]:
import sys
from collections import defaultdict
sys.path.insert(0, "../pipeline")
import build_dataset as bd
from transformers import AutoTokenizer

tok_hf = AutoTokenizer.from_pretrained(bd.TOKENIZER)
n_tokens = lambda text: len(tok_hf(text)["input_ids"])
rng = random.Random(bd.SEED)
# session/ was built from the v1 turns (17 Oct); data/processed now holds the newer copy
sesiones, chat_sesiones = bd.load_sessions(bd.ROOT / "data" / "processed_v1_2026-10-17" / "turns.jsonl")
split = bd.assign_splits(chat_sesiones, rng)

sesion_mia, sesion_amigos = [], []
for s in sorted(sesiones):
    if split[s] != "test":
        continue
    d = defaultdict(int)
    sesion_mia += [s] * len(bd.make_examples(sesiones[s], 1, d, n_tokens))
    sesion_amigos += [s] * len(bd.make_examples(sesiones[s], 0, d, n_tokens))

# Test sessions with at least one example (mine or friends'): what the bootstrap resamples
sesiones_test = sorted(set(sesion_mia) | set(sesion_amigos))

print(len(sesion_mia), len(sesion_amigos), len(set(sesion_mia)), len(sesiones_test))  # 1133, 1110, 63 with messages of mine
assert (len(sesion_mia), len(sesion_amigos)) == (1133, 1110), "sessions don't match the test files"

1133 1110 63 67


In [8]:
def medir(model, tokenizer, ruta):
    with open(ruta, "r") as f:
        ejemplos = [json.loads(line) for line in f if line.strip()]
    return [nll_ejemplo(model, tokenizer, e["messages"]) for e in ejemplos]

In [9]:
def por_sesion(resultados, etiquetas):
    sumas = {}
    for (nll, n), s in zip(resultados, etiquetas):
        if s not in sumas:
            sumas[s] = [0.0, 0]
        sumas[s][0] += nll
        sumas[s][1] += n
    return sumas

In [10]:
def perdida(sumas, elegidas):
    nll, n = 0.0, 0
    for s in elegidas:
        nll+= sumas.get(s, [0.0, 0])[0]
        n+= sumas.get(s, [0.0, 0])[1]
    return nll/n

In [11]:
# The slow part: 4 passes through the models. Done once and saved to disk.
CACHE = "results/perplexity_final_adapter.json"
MIA = "../data/dataset/session/test.jsonl"
AMIGOS = "../data/dataset/session/test_friends.jsonl"

try:
    with open(CACHE) as f:
        r = json.load(f)
except FileNotFoundError:
    r = {
        "r1_mia": medir(model, tokenizer, MIA),
        "r1_amigos": medir(model, tokenizer, AMIGOS),
        "b_mia": medir(base, tok_b, MIA),
        "b_amigos": medir(base, tok_b, AMIGOS),
    }
    with open(CACHE, "w") as f:
        json.dump(r, f)

print({k: len(v) for k, v in r.items()})  # 1133 and 1110

{'r1_mia': 1133, 'r1_amigos': 1110, 'b_mia': 1133, 'b_amigos': 1110}


In [12]:
s_r1_mia    = por_sesion(r["r1_mia"], sesion_mia)
s_r1_amigos = por_sesion(r["r1_amigos"], sesion_amigos)
s_b_mia     = por_sesion(r["b_mia"], sesion_mia)
s_b_amigos  = por_sesion(r["b_amigos"], sesion_amigos)

# Check: with every session once it should give 3.388 (final adapter; run1 gave 3.4214)
print(perdida(s_r1_mia, sesiones_test))

3.3881103444665945


In [13]:
def senal(elegidas):
    ppl_r1_mia     = math.exp(perdida(s_r1_mia, elegidas))
    ppl_b_mia      = math.exp(perdida(s_b_mia, elegidas))
    ppl_r1_amigos  = math.exp(perdida(s_r1_amigos, elegidas))
    ppl_b_amigos   = math.exp(perdida(s_b_amigos, elegidas))

    bajada_mia = 1 - ppl_r1_mia / ppl_b_mia
    bajada_amigos = 1 - ppl_r1_amigos / ppl_b_amigos
    return bajada_mia - bajada_amigos

# Signal with the whole test, no resampling (mlx's measure gave 8.1 points, biased by the padding bug)
print(senal(sesiones_test))

0.19824140291859904


In [14]:
import numpy as np
resultados = []
random.seed(0)
for i in range(1000):
    elegidas = random.choices(sesiones_test, k=len(sesiones_test))
    resultados.append(senal(elegidas))

np.percentile(resultados, [2.5, 97.5])

array([0.14679978, 0.25825613])

## Bootstrap results

The results were better than expected, for two reasons.

**1. A possible bug in the library.** `mlx_lm` computes the loss counting one token too many: the first padding token that comes right after the end of the response (a `!`). This makes the base model look worse than it is, because it doesn't expect that `!`, and the LoRA-adapted one look better, because it has learned to predict it. I reported it in [ml-explore/mlx-lm#2029](https://github.com/ml-explore/mlx-lm/issues/2029). Here I measured the loss by hand with `nll_ejemplo`, counting only the response tokens.

| Measure on test (my responses) | With mlx (bug) | By hand (correct) |
|---|---|---|
| Base perplexity | 209.2 | **75.5** |
| Final adapter perplexity | 24.7 | **29.6** |

**2. The difference between my messages and my friends' is bigger than it looked.** With the corrected numbers:

| Responses from | Base PPL | Final PPL | Drop | run1 PPL (1,000 steps) | run1 drop |
|---|---|---|---|---|---|
| Me (test) | 75.5 | 29.6 | 60.8% | 30.6 | 59.5% |
| My friends (test_friends) | 81.1 | 47.9 | 41.0% | 47.0 | 42.0% |
| **Signal (me - friends)** | | | **19.8 points** | | 17.4 points |

With mlx's numbers the signal was 8.1 points (7.4 for run1): the bug inflated both drops and hid more than half of the difference.

**3. Bootstrap.** Resampling the test sessions 1000 times (with replacement, seed 0), the signal holds: the perplexity drop on my responses is **19.8 points larger** than on my friends', with a **95% confidence interval from 14.7 to 25.8 points** (run1: 17.4, from 12.1 to 23.3). The interval doesn't touch zero, so it's solid: the model hasn't only learned to talk the way people talk on WhatsApp, it has also learned something of how I talk. Training longer (run1 to the final adapter) makes it a bit more mine and a bit less my friends'.
